# 10 - Real IBM hardware for the advantage test (week 3)
Owner: **Farrukh**

Professor's feedback: *"this is a QML setting that is ripe for quantum advantage; let us see if you see something."* Our only advantage so far is the projected kernel on **engineered labels at $n=6$** (notebook 05), and only in perfect simulation. Notebook 11 tests whether it survives the device. This notebook collects the hardware data it needs.

- **What runs:** the projected kernel's 3 basis circuits for all 120 images (80 train + 40 test) + 2 readout-calibration circuits = **362 circuits, 2048 shots**, for **seeds 0 and 100** (the first two seeds), $n=6$, on `ibm_kingston`. Same transpilation and physical qubits 0-5 as notebooks 06 and 07.
- **Why $\alpha = 1$ (new run, not notebook 07's data):** notebook 07 ran seed 0 at $\alpha=2$, the setting tuned on the *real* labels. On the engineered labels, inner-CV tuning chose $\alpha=1$ in **all 15** $n=6$ runs (5 seeds $\times$ 3 $\lambda$), and the labels themselves are generated with $\alpha=1$. Hardware can afford only one $\alpha$, so it is $\alpha=1$.
- **Labels are not needed here:** the circuits depend only on the images. Notebook 11 attaches the engineered labels afterwards.
- **Quota:** Open Plan = 600 s per 28 days; notebook 07 used 197 s. Each job is submitted **one at a time**, and only if the remaining quota (read from IBM) covers it; otherwise the notebook stops and we keep what we have.
- **Safety:** dry run on the simulated `ibm_kingston` first (no quota). Job IDs and raw counts are saved immediately; rerunning never resubmits.

In [1]:
import sys, json, time
sys.path.insert(0, "..")
import numpy as np
from qiskit_ibm_runtime.fake_provider import FakeKingston
from src.common import get_split, make_preprocessor, encoding_circuit, result_path, SHOTS
from src.projected import pauli_features_exact
from src.hardware import (BACKEND_NAME, build_circuits, transpile_for, dry_run_counts, submit,
                          counts_from_result, features_from_counts, readout_errors,
                          mitigate_readout, save_counts, load_counts)

## 1. Settings

In [ ]:
HW_SEEDS = [0, 100]            # first two seeds
N, LAYERS, ALPHA = 6, 2, 1.0
RUN_HARDWARE = False           # set to True ONLY for the real run (section 4)
MIN_QUOTA_S = 210              # notebook 07's job used 197 s; do not submit with less left

for s in HW_SEEDS:             # confirm alpha = 1 is what tuning chose on the engineered labels
    chosen = [json.loads(result_path("tables", f"engineered_lam{t}", s, N, "json").read_text())
              ["projected"]["best"]["alpha"] for t in ("0p1", "0p01", "0p001")]
    print(f"seed {s}: alpha chosen on engineered labels (lambda 0.1, 0.01, 0.001) = {chosen}")

seed 0: alpha chosen on engineered labels (lambda 0.1, 0.01, 0.001) = [1.0, 1.0, 1.0]
seed 100: alpha chosen on engineered labels (lambda 0.1, 0.01, 0.001) = [1.0, 1.0, 1.0]


## 2. Circuits (362 per seed)

In [3]:
fake = FakeKingston()
enc = encoding_circuit(N, LAYERS)
data = {}
for s in HW_SEEDS:
    Xtr, ytr, Xte, yte = get_split(s)
    tf = make_preprocessor(N, ALPHA)(Xtr)               # fitted on training images only
    A_all = np.vstack([tf(Xtr), tf(Xte)])               # 80 train + 40 test images
    circs = build_circuits(enc, A_all)
    data[s] = dict(A_all=A_all, circs=circs, tc_fake=transpile_for(fake, circs, N),
                   F_exact=pauli_features_exact(enc, A_all))
    t = data[s]["tc_fake"][2]
    print(f"seed {s}: {len(circs)} circuits; transpiled basis circuit depth {t.depth()}, "
          f"{t.count_ops().get('cz', 0)} CZ, physical qubits {list(t.layout.final_index_layout())}")
n_all = len(data[HW_SEEDS[0]]["A_all"])

def split_counts(counts):
    return counts[:3 * n_all], counts[3 * n_all], counts[3 * n_all + 1]

def summary(counts, F_exact):
    c_basis, c0, c1 = split_counts(counts)
    F = features_from_counts(c_basis, n_all, N)
    p10, p01 = readout_errors(c0, c1, N)
    Fm = mitigate_readout(F, p10, p01)
    return dict(F=F, p10=p10, p01=p01,
                err=float(np.abs(F - F_exact).mean()), err_mit=float(np.abs(Fm - F_exact).mean()),
                bloch=float(np.linalg.norm(F, axis=-1).mean()),
                bloch_mit=float(np.linalg.norm(Fm, axis=-1).mean()),
                bloch_exact=float(np.linalg.norm(F_exact, axis=-1).mean()),
                mean_Y=float(F[..., 1].mean()))

def show(name, r):
    print(f"{name}: feature error vs exact {r['err']:.3f} (mitigated {r['err_mit']:.3f}); "
          f"Bloch length {r['bloch']:.3f} (mitigated {r['bloch_mit']:.3f}, exact {r['bloch_exact']:.3f}); "
          f"mean <Y> {r['mean_Y']:+.4f}")
    print(f"   readout p(1|0) {np.round(r['p10'], 3)}   p(0|1) {np.round(r['p01'], 3)}")

seed 0: 362 circuits; transpiled basis circuit depth 33, 10 CZ, physical qubits [0, 1, 2, 3, 4, 5]
seed 100: 362 circuits; transpiled basis circuit depth 33, 10 CZ, physical qubits [0, 1, 2, 3, 4, 5]


## 3. Dry run on the simulated `ibm_kingston` (no quota used)
The counts are saved; notebook 11 reuses them as its noise-model level for these seeds.

In [4]:
for s in HW_SEEDS:
    path = result_path("hardware", "dryrun_counts_a1", s, N, "json")
    if path.exists():
        counts = load_counts(path)[3]
    else:
        t0 = time.time()
        counts = dry_run_counts(fake, data[s]["tc_fake"], seed=s)
        save_counts(path, "local-dry-run", "FakeKingston", counts, dict(seed=s, n=N, alpha=ALPHA, shots=SHOTS))
        print(f"seed {s}: dry run {time.time() - t0:.0f} s, saved {path.name}")
    show(f"seed {s} dry run", summary(counts, data[s]["F_exact"]))

seed 0: dry run 13 s, saved dryrun_counts_a1_seed0_q6.json
seed 0 dry run: feature error vs exact 0.033 (mitigated 0.019); Bloch length 0.632 (mitigated 0.681, exact 0.690); mean <Y> +0.0002
   readout p(1|0) [0.011 0.166 0.009 0.008 0.011 0.007]   p(0|1) [0.011 0.183 0.01  0.007 0.011 0.007]
seed 100: dry run 13 s, saved dryrun_counts_a1_seed100_q6.json
seed 100 dry run: feature error vs exact 0.033 (mitigated 0.020); Bloch length 0.542 (mitigated 0.590, exact 0.593); mean <Y> +0.0009
   readout p(1|0) [0.011 0.172 0.005 0.006 0.014 0.007]   p(0|1) [0.012 0.19  0.009 0.006 0.015 0.005]


## 4. Real hardware run on `ibm_kingston`

Set `RUN_HARDWARE = True` in section 1, rerun section 1, then run this cell. For each seed in turn: if its counts file exists it is loaded; if only its job ID exists the finished job is fetched; otherwise the job is submitted **only if** IBM reports at least `MIN_QUOTA_S` seconds left, and the cell waits for it before moving to the next seed. Safe to rerun at any time.

In [8]:
from qiskit_ibm_runtime import QiskitRuntimeService

def remaining_quota(service):
    try:
        u = service.usage()
        return u.get("usage_remaining_seconds"), u
    except Exception as e:
        return None, f"usage not available: {e}"

service = backend = None
for s in HW_SEEDS:
    hw_path = result_path("hardware", "hw_counts_a1", s, N, "json")
    id_path = result_path("hardware", "hw_job_id_a1", s, N, "txt")
    if hw_path.exists():
        print(f"seed {s}: loaded saved hardware counts (job {load_counts(hw_path)[0]}) - nothing submitted")
        continue
    if not (id_path.exists() or RUN_HARDWARE):
        print(f"seed {s}: RUN_HARDWARE is False - nothing submitted.")
        continue
    service = service or QiskitRuntimeService()
    if id_path.exists():
        job_id = id_path.read_text().strip()
        job = service.job(job_id)
        print(f"seed {s}: fetching existing job {job_id}, status {job.status()} (waits if still queued)")
    else:
        left, info = remaining_quota(service)
        print(f"seed {s}: remaining quota {left} s  ({info})")
        if left is None or left < MIN_QUOTA_S:
            print(f"seed {s}: NOT submitted - not enough quota known to be left. Stopping here.")
            break
        backend = backend or service.backend(BACKEND_NAME)
        tc_hw = transpile_for(backend, data[s]["circs"], N)
        job = submit(backend, tc_hw)
        job_id = job.job_id()
        id_path.write_text(job_id)                       # saved before waiting
        print(f"seed {s}: submitted {len(tc_hw)} circuits x {SHOTS} shots, job {job_id} (ID saved); waiting ...")
    t0 = time.time()
    counts = counts_from_result(job.result(), len(data[s]["circs"]))
    save_counts(hw_path, job_id, BACKEND_NAME, counts, dict(seed=s, n=N, alpha=ALPHA, shots=SHOTS))
    print(f"seed {s}: done after {time.time() - t0:.0f} s, saved {hw_path.name}")
    try:
        print(f"seed {s}: quantum time used by this job (s):", job.usage())
    except Exception as e:
        print("usage not reported:", e)

seed 0: remaining quota 401 s  ({'instance_id': 'crn:v1:bluemix:public:quantum-computing:us-east:a/e7f7cf9f1934431385e59de6e2329233:12daaa30-ef6f-47b2-8959-eff77d73108b::', 'plan_id': '850b21a7-71de-4e53-9441-1abdd202f35d', 'usage_consumed_seconds': 199, 'usage_consumed_ru': 3.309924, 'usage_period': {'start_time': '2026-10-28T10:50:02.427Z', 'end_time': '2026-11-25T10:50:02.427Z'}, 'usage_limit_seconds': 600, 'usage_limit_ru': 10, 'usage_limit_reached': False, 'usage_remaining_seconds': 401})
seed 0: submitted 362 circuits x 2048 shots, job dc3bqo5hdvas73bniep0 (ID saved); waiting ...
seed 0: done after 240 s, saved hw_counts_a1_seed0_q6.json
seed 0: quantum time used by this job (s): 197
seed 100: remaining quota 204 s  ({'instance_id': 'crn:v1:bluemix:public:quantum-computing:us-east:a/e7f7cf9f1934431385e59de6e2329233:12daaa30-ef6f-47b2-8959-eff77d73108b::', 'plan_id': '850b21a7-71de-4e53-9441-1abdd202f35d', 'usage_consumed_seconds': 396, 'usage_consumed_ru': 6.587131, 'usage_period

## 5. Hardware features: quick check
Raw and readout-mitigated features against the exact ones (mitigation uses each job's own 2 calibration circuits, `hardware.mitigate_readout`). The features are saved for notebook 11.

In [9]:
for s in HW_SEEDS:
    hw_path = result_path("hardware", "hw_counts_a1", s, N, "json")
    if not hw_path.exists():
        print(f"seed {s}: no hardware counts yet")
        continue
    job_id, backend_name, meta, counts = load_counts(hw_path)
    r = summary(counts, data[s]["F_exact"])
    np.save(result_path("features", "pauli_hardware_a1_all", s, N, "npy"), r["F"])
    show(f"seed {s} {backend_name} (job {job_id})", r)

seed 0 ibm_kingston (job dc3bqo5hdvas73bniep0): feature error vs exact 0.023 (mitigated 0.023); Bloch length 0.668 (mitigated 0.670, exact 0.690); mean <Y> +0.0093
   readout p(1|0) [0.006 0.001 0.007 0.004 0.002 0.004]   p(0|1) [0.016 0.02  0.016 0.015 0.011 0.01 ]
seed 100: no hardware counts yet
